# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² project dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}")
print(f"Dataset version: {metadata.version}")
print(f"Date published: {metadata.datePublished}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review available record sets, fields, and their IDs. We use the Croissant `@id` for consistent referencing.

In [ ]:
# List record sets and their fields with @id references
from mlcroissant.types import RecordSet, Field

record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in the dataset metadata. Trying to auto-extract from dataset...\n")
    # Try to infer from dataset object
    # mlcroissant populates available record set ids via dataset.record_sets
    inferred_record_sets = getattr(dataset, 'record_sets', [])
    if not inferred_record_sets:
        print("No record sets available in this schema.")
    else:
        print(f"Discovered {len(inferred_record_sets)} record sets:")
        for idx, rs in enumerate(inferred_record_sets):
            print(f"{idx+1}. @id: {rs['@id']}, name: {rs.get('name', '<no_name>')}")
else:
    print(f"Found {len(record_sets)} record sets in the metadata:")
    for idx, rs in enumerate(record_sets):
        if isinstance(rs, dict):
            rs_id = rs.get('@id', str(rs))
            name = rs.get('name', '<no_name>')
        else:
            rs_id = str(rs)
            name = '<no_name>'
        print(f"{idx+1}. @id: {rs_id}, name: {name}")

# Attempt to list all record set IDs
record_set_ids = []
if getattr(dataset, 'record_sets', None):
    for rs in dataset.record_sets:
        if '@id' in rs:
            record_set_ids.append(rs['@id'])
    print("\nAll record set @id values:")
    print(record_set_ids)

# For demonstration, print the fields in the first record set
if record_set_ids:
    print(f"\nFields for record set @id='{record_set_ids[0]}':")
    try:
        fields = dataset.get_record_set(record_set_ids[0])["field"]
        for idx, field in enumerate(fields):
            print(f"{idx+1}. Field @id: {field['@id']}, name: {field.get('name', '<no_name>')} (type: {field.get('dataType', '<no_type>')})")
    except Exception as e:
        print(f"Could not retrieve fields for record set {record_set_ids[0]}: {str(e)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all available record sets, referenced by @id
import itertools

# If no record set IDs discovered, try to extract from mlcroissant API
if not record_set_ids:
    # Try to auto-infer
    record_set_ids = [rs['@id'] for rs in getattr(dataset, 'record_sets', [])]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records for record set '{record_set_id}'")
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {str(e)}")

# For illustration, preview one DataFrame
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    # Show a few rows
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes.

Replace `numeric_field_id` and `group_field_id` with actual `@id` values from the previous cell.

In [ ]:
# EDA: Filter, normalize, group by (replace IDs with actual `@id`s from data overview)
import numpy as np

# Select a target record set for analysis
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Find a plausible numeric field (by scanning columns with float or int type after converting)
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(pd.to_numeric(df[col], errors='coerce'))]
print(f"Potential numeric fields: {numeric_field_candidates}")
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]  # pick one (replace with your preferred field)
else:
    print("No numeric field found.")
    numeric_field = None

if numeric_field:
    # Make sure the column is float for filter & math
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    # Remove NaN
    df_clean = df.dropna(subset=[numeric_field])
    # Filter: e.g., values > threshold
    threshold = df_clean[numeric_field].mean() if pd.notnull(df_clean[numeric_field].mean()) else 0
    filtered_df = df_clean[df_clean[numeric_field] > threshold]
    print(f"Filtered records with '{numeric_field}' > {threshold:.2f} (mean)")
    print(filtered_df.head())

    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, normalized_col]].head())

    # Try to group by a non-numeric/likely categorical field, if available
    group_field_candidates = [col for col in df.columns if col != numeric_field]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        if group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
            print(grouped_df.head())
        else:
            print(f"{group_field} not in columns.")
    else:
        print("No non-numeric group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization: Histogram and Boxplot of the main numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f'Boxplot of {numeric_field}')
    plt.xlabel(numeric_field)

    plt.tight_layout()
    plt.show()

    # If we did grouping, vis group means
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field])
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.ylabel(f'Mean {numeric_field}')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook provided stepwise loading, exploration, and preliminary analysis of the FAIR² mass spectrometry extracellular vesicle dataset, using all references by their Croissant `@id`. Use this framework to further analyze or model the dataset, replacing variable selections as appropriate for your scientific questions.